In [1]:
import shap
import xgboost
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('nifty_clean.csv', index_col=0, low_memory=False)

In [3]:
len(df)

2945

In [4]:
df.shape

(2945, 110)

In [5]:
a=[]
a=df.columns

In [6]:
# Drop irrelevant columns
X = df.drop(columns=["date", "Signal","SUPERTl_7_3.0","SUPERTs_7_3.0"])  # Keep 100 indicators
y = df["Signal"]  # Target labels (Buy, Sell, Hold)

In [7]:
# Check for NaN or Inf values
summary = df.isna().sum().to_frame(name="NaN Count")  # Count of NaN values
summary["Inf Count"] = ((df == np.inf) | (df == -np.inf)).sum()  # Count of Inf values

print(summary)

        NaN Count  Inf Count
Close           0          0
High            0          0
Low             0          0
Open            0          0
Volume          0          0
...           ...        ...
CS2             0          0
CR3             0          0
CS3             0          0
CR4             0          0
CS4             0          0

[110 rows x 2 columns]


In [8]:
# Skip the first 200 rows
X_new = X.iloc[200:]
y_new = y.iloc[200:]

In [9]:
a=[]
a=X_new.columns

In [10]:
import numpy as np
print(np.any(np.isinf(X_new)))  # Check for infinite values
print(np.any(np.isnan(X_new)))  # Check for NaN values
print(np.max(X_new))  # Check for extremely large values

True
True
inf


In [11]:
total_nan_count = X_new.isna().sum().sum()
print("Total NaN count:", total_nan_count)

Total NaN count: 5


In [12]:
total_inf_count = np.isinf(X_new).sum().sum()
print("Total inf count:", total_inf_count)

Total inf count: 52


In [13]:
print(np.any(np.isinf(y_new)))  # Check for infinite values
print(np.any(np.isnan(y_new)))  # Check for NaN values
print(np.max(y_new))  # Check for extremely large values

False
False
1


In [14]:
# Get the indices for NaN values
nan_rows, nan_cols = np.where(X_new.isna())

# Loop through and print the row index and corresponding column name for each NaN
for row, col in zip(nan_rows, nan_cols):
    print(f"Row: {row}, Column: {df.columns[col]}")

Row: 1175, Column: CMF_20
Row: 1195, Column: CMF_20
Row: 1223, Column: AD
Row: 1243, Column: CMF_20
Row: 2744, Column: ol


In [15]:
X_new = np.where(np.isinf(X_new), np.nan, X_new)  # Replace inf with NaN
X_new = np.nan_to_num(X_new, nan=0.0)  # Replace NaN with 0

In [16]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_new, y_new, test_size=0.2, random_state=42)

In [17]:
# Train an XGBoost model for classification
model = xgboost.XGBClassifier(n_estimators=1000, learning_rate=0.1, max_depth=5)
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

In [18]:
y_pred = model.predict(X_test)

In [19]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

# Display a detailed classification report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Optionally, display the confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Test Accuracy: 0.5610200364298725
Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.64      0.60       284
           1       0.55      0.47      0.51       265

    accuracy                           0.56       549
   macro avg       0.56      0.56      0.56       549
weighted avg       0.56      0.56      0.56       549

Confusion Matrix:
[[183 101]
 [140 125]]


In [20]:
# Compute SHAP values
explainer = shap.Explainer(model, X_train)
shap_values = explainer(X_train)

100%|===================| 2192/2196 [01:11<00:00]        

In [22]:
print(shap_values.values.shape)  # Should be (num_samples, num_features)

(2196, 106)


In [23]:

# Suppose these are your feature names; make sure the number of names matches the number of columns in X_train.
feature_names = a  # Replace with your actual feature names

# Convert the NumPy array to a DataFrame
X_train_df = pd.DataFrame(X_train, columns=feature_names)

# Now you can create the SHAP importance DataFrame using the DataFrame's columns attribute
shap_importance = pd.DataFrame({
    "Feature": X_train_df.columns,
    "SHAP Importance": np.abs(shap_values.values).mean(axis=0)  # Mean over samples & classes
}).sort_values(by="SHAP Importance", ascending=False)

print(shap_importance)


        Feature  SHAP Importance
70  Log_Returns         0.577760
7            oc         0.400696
47       MOM_10         0.366635
43       ROC_10         0.357718
10           ol         0.318495
..          ...              ...
54    BB_Middle         0.000000
99          CS1         0.000000
48       CMO_14         0.000000
60     KC_Upper         0.000000
96          FR3         0.000000

[106 rows x 2 columns]


In [24]:
# Save selected indicators
shap_importance.to_csv("top_indicators.csv", index=False)

for plots

In [25]:
explainer = shap.TreeExplainer(model)  # Use TreeExplainer for tree-based models
shap_values = explainer.shap_values(X_test)  # Get SHAP values

In [26]:
print("SHAP values shape:", shap_values.shape)
print("X_test shape:", X_test.shape)

SHAP values shape: (549, 106)
X_test shape: (549, 106)


In [27]:
import matplotlib
matplotlib.use('Agg')  # Set a non-interactive backend

In [29]:
import shap
import matplotlib.pyplot as plt

shap.summary_plot(shap_values, X_test)

plt.savefig("shap_summary_nifty10.png", dpi=300, bbox_inches='tight')
plt.close()

C:\Users\IIITNAGPUR\anaconda3\Lib\site-packages\shap\plots\_beeswarm.py:952: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pl.show()


In [31]:
import shap
import matplotlib.pyplot as plt

# Generate SHAP Bar Plot
shap.summary_plot(shap_values, X_test, plot_type="bar")

# Get the current figure and force rendering
fig = plt.gcf()
fig.canvas.draw()

# Save the figure
fig.savefig("shap_bar_plot_nifty10.png", dpi=300, bbox_inches='tight')

# Close the figure
plt.close()


C:\Users\IIITNAGPUR\anaconda3\Lib\site-packages\shap\plots\_beeswarm.py:952: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pl.show()
